# Gold Fact: TfL Arrival Observations

Build the historical Tube arrival-observation fact table.

**Source:** `workspace.urbanpulse_silver.tfl_arrivals`

**Target:** `workspace.urbanpulse_gold.fact_arrival_observation`

**Grain:** One arrival prediction observed during one TfL polling request.

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import dependencies

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.fact_arrival_observation import (
    build_fact_arrival_observation,
)

from urbanpulse.quality.fact_arrival_observation import (
    invalid_fact_arrival_observation,
)

from urbanpulse.utils.delta import (
    merge_insert_only,
)

## 3. Define tables

In [0]:
SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "tfl_arrivals"
)

DIM_STATION = (
    "workspace."
    "urbanpulse_gold."
    "dim_station"
)

DIM_LINE = (
    "workspace."
    "urbanpulse_gold."
    "dim_line"
)

DIM_DATE = (
    "workspace."
    "urbanpulse_gold."
    "dim_date"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "fact_arrival_observation"
)

## 4. Read source and dimensions

In [0]:
silver_df = spark.table(
    SILVER_TABLE
)

dim_station_df = spark.table(
    DIM_STATION
)

dim_line_df = spark.table(
    DIM_LINE
)

dim_date_df = spark.table(
    DIM_DATE
)

source_count = silver_df.count()

print(
    f"Silver arrival observations: "
    f"{source_count}"
)

## 5. Validate Silver observation grain

`arrival_observation_key` must uniquely identify each observed arrival prediction.

In [0]:
duplicate_source_keys_df = (
    silver_df
    .groupBy(
        "arrival_observation_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_source_keys_df.count() > 0:
    display(
        duplicate_source_keys_df
    )

    raise ValueError(
        "Duplicate Silver arrival "
        "observation keys detected."
    )

print(
    "Silver arrival grain validation passed."
)

## 6. Resolve dimensions

Each arrival observation is resolved to the station and line versions valid at the prediction timestamp, plus its London-local calendar date.

In [0]:
fact_df = (
    build_fact_arrival_observation(
        silver_df=silver_df,
        dim_station_df=dim_station_df,
        dim_line_df=dim_line_df,
        dim_date_df=dim_date_df,
    )
)

fact_count = fact_df.count()

print(
    f"Silver rows: {source_count}"
)

print(
    f"Fact rows:   {fact_count}"
)

display(
    fact_df
    .orderBy(
        F.col(
            "prediction_timestamp"
        ).desc()
    )
)

## 7. Apply fact quality checks

In [0]:
invalid_df = (
    invalid_fact_arrival_observation(
        fact_df
    )
)

invalid_count = invalid_df.count()

print(
    f"Invalid facts: {invalid_count}"
)

if invalid_count > 0:
    display(invalid_df)

    raise ValueError(
        f"{invalid_count} invalid "
        "arrival facts detected."
    )

print(
    "Arrival fact quality checks passed."
)

In [0]:
# Validate Fact-key uniqueness

duplicate_fact_keys_df = (
    fact_df
    .groupBy(
        "arrival_observation_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_fact_keys_df.count() > 0:
    display(
        duplicate_fact_keys_df
    )

    raise ValueError(
        "Duplicate arrival fact keys detected."
    )

print(
    "Arrival fact keys are unique."
)

In [0]:
# Add gold metadata

gold_df = (
    fact_df
    .withColumn(
        "created_at",
        F.current_timestamp(),
    )
)

## 8. Merge arrival observations into Gold

Arrival facts are insert-only. Historical observations are never overwritten.

In [0]:
merge_insert_only(
    spark=spark,
    source_df=gold_df,
    target_table=TARGET_TABLE,
    merge_condition="""
        target.arrival_observation_key
        =
        source.arrival_observation_key
    """,
)

print(
    f"Gold fact updated: "
    f"{TARGET_TABLE}"
)

In [0]:
%sql
-- Verify Fact table

SELECT
    arrival_observation_key,
    station_key,
    line_key,
    requested_station_id,
    line_id,
    prediction_date_key,
    prediction_timestamp,
    expected_arrival,
    time_to_station_seconds,
    vehicle_id,
    platform_name
FROM workspace.urbanpulse_gold.fact_arrival_observation
ORDER BY prediction_timestamp DESC;

In [0]:
%sql
-- Verify station joins
SELECT
    s.station_name,
    l.line_name,
    f.platform_name,
    f.destination_name,
    f.time_to_station_seconds,
    f.expected_arrival,
    f.prediction_timestamp
FROM workspace.urbanpulse_gold.fact_arrival_observation f

INNER JOIN workspace.urbanpulse_gold.dim_station s
    ON f.station_key = s.station_key

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key

ORDER BY
    f.prediction_timestamp DESC;

In [0]:
%sql 
-- Referential Integrity Check
SELECT f.*
FROM workspace.urbanpulse_gold.fact_arrival_observation f

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_station s
    ON f.station_key = s.station_key;

In [0]:
%sql
SELECT f.*
FROM workspace.urbanpulse_gold.fact_arrival_observation f

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key;

In [0]:
%sql
SELECT f.*
FROM workspace.urbanpulse_gold.fact_arrival_observation f

LEFT ANTI JOIN workspace.urbanpulse_gold.dim_date d
    ON f.prediction_date_key = d.date_key;

In [0]:
%sql
-- Validate Station SCD joins -> expecting 0 rows
SELECT
    f.arrival_observation_key,
    f.requested_station_id,
    f.prediction_timestamp,
    s.effective_from,
    s.effective_to
FROM workspace.urbanpulse_gold.fact_arrival_observation f

INNER JOIN workspace.urbanpulse_gold.dim_station s
    ON f.station_key = s.station_key

WHERE
    f.prediction_timestamp < s.effective_from

    OR (
        s.effective_to IS NOT NULL
        AND f.prediction_timestamp >= s.effective_to
    );

In [0]:
%sql
-- Validate Line SCD joins -> expecting 0 rows
SELECT
    f.arrival_observation_key,
    f.line_id,
    f.prediction_timestamp,
    l.effective_from,
    l.effective_to
FROM workspace.urbanpulse_gold.fact_arrival_observation f

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key

WHERE
    f.prediction_timestamp < l.effective_from

    OR (
        l.effective_to IS NOT NULL
        AND f.prediction_timestamp >= l.effective_to
    );

In [0]:
%sql
-- Avg. predicted ETA by station and line
SELECT
    s.station_name,
    l.line_name,
    COUNT(*) AS observations,
    ROUND(
        AVG(f.time_to_station_seconds),
        1
    ) AS avg_eta_seconds,
    MIN(
        f.time_to_station_seconds
    ) AS min_eta_seconds,
    MAX(
        f.time_to_station_seconds
    ) AS max_eta_seconds

FROM workspace.urbanpulse_gold.fact_arrival_observation f

INNER JOIN workspace.urbanpulse_gold.dim_station s
    ON f.station_key = s.station_key

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON f.line_key = l.line_key

GROUP BY
    s.station_key,
    s.station_name,
    l.line_key,
    l.line_name

ORDER BY
    observations DESC;

In [0]:
%sql
-- Distinct observed vehicles
SELECT
    s.station_name,
    COUNT(
        DISTINCT f.vehicle_id
    ) AS observed_vehicles,
    COUNT(*) AS arrival_observations

FROM workspace.urbanpulse_gold.fact_arrival_observation f

INNER JOIN workspace.urbanpulse_gold.dim_station s
    ON f.station_key = s.station_key

GROUP BY
    s.station_key,
    s.station_name

ORDER BY
    arrival_observations DESC;

In [0]:
%sql
SELECT COUNT(*) AS facts
FROM workspace.urbanpulse_gold.fact_arrival_observation;